# Chicago Bike Network: Operations Dashboard
Live data from the AWS pipeline (EventBridge → Lambda → S3 → Glue Catalog → Athena). Re-run all cells to refresh.

In [1]:
BUCKET  = "nrt-bike-nav-123456"          # S3 bucket that holds the data lake
REGION  = "us-east-2"
PROFILE = "bike-notebook"

from datetime import datetime, timezone

import awswrangler as wr
import boto3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DB, OUTPUT = "bike_db", f"s3://{BUCKET}/athena-results/"
TODAY = datetime.now(timezone.utc).strftime("%Y-%m-%d")        # partitions are UTC
LOCAL_TZ, MIN_CAPACITY, TEMPLATE = "America/Chicago", 15, "plotly_white"

# Corrals (event overflow parking) and public racks are parking-only: they are
# supposed to show 0 bikes, so they are excluded from operational metrics.
VIRTUAL_TERMS = ["Corral", "Public Rack"]
REAL_SQL = " AND ".join(f"name NOT LIKE '%{t}%'" for t in VIRTUAL_TERMS)
VIRTUAL_RE = "|".join(VIRTUAL_TERMS)

session = boto3.Session(profile_name=PROFILE, region_name=REGION)


def q(sql: str, name: str) -> pd.DataFrame:
    """Run an Athena query (tagged with its name in query history) and return a DataFrame."""
    sql = f"-- q:{name}\n" + sql.format(today=TODAY, min_cap=MIN_CAPACITY, real=REAL_SQL)
    return wr.athena.read_sql_query(sql, database=DB, s3_output=OUTPUT,
                                    ctas_approach=False, boto3_session=session)

## Pipeline health

In [2]:
fresh = q("""
SELECT count(DISTINCT ingest_ts) AS snapshots, count(*) AS rows_ingested, max(ingest_ts) AS latest_ts
FROM bike_db.station_status WHERE dt = '{today}'
""", "freshness").iloc[0]

if int(fresh["snapshots"]) == 0:
    raise RuntimeError(f"No data for {TODAY}. Check the EventBridge schedule and the ingest Lambda logs.")

latest_ts = datetime.fromtimestamp(int(fresh["latest_ts"]), timezone.utc)
age_min = (datetime.now(timezone.utc) - latest_ts).total_seconds() / 60
as_of = pd.Timestamp(latest_ts).tz_convert(LOCAL_TZ).strftime("%Y-%m-%d %H:%M %Z")

print(f"{'HEALTHY' if age_min < 15 else 'STALE: check the ingest Lambda'}  |  "
      f"{int(fresh['snapshots'])} snapshots today, {int(fresh['rows_ingested']):,} rows  |  "
      f"latest {as_of} ({age_min:.0f} min ago)")

HEALTHY  |  34 snapshots today, 70,686 rows  |  latest 2026-11-05 16:36 CST (3 min ago)


In [3]:
latest = q("""
SELECT name, lat, lon, capacity, num_bikes_available, num_ebikes_available,
       num_docks_available, is_renting, is_returning, pct_full, ingest_ts, last_reported
FROM bike_db.v_station_health
WHERE dt = '{today}'
  AND ingest_ts = (SELECT max(ingest_ts) FROM bike_db.station_status WHERE dt = '{today}')
""", "latest_snapshot")

latest["is_virtual"] = latest["name"].str.contains(VIRTUAL_RE, regex=True)
latest["pct_full"] = latest["pct_full"].clip(0, 100)       # a few stations report more bikes than docks
latest["state"] = "OK"
latest.loc[latest["pct_full"] < 20, "state"] = "Low"
latest.loc[latest["num_docks_available"] == 0, "state"] = "Full"
latest.loc[latest["num_bikes_available"] == 0, "state"] = "Empty"

stations = latest[~latest["is_virtual"]]                       # real docking stations
renting = stations[stations["is_renting"] == 1]                # open for rentals
large = renting[renting["capacity"] >= MIN_CAPACITY]

In [4]:
bikes, ebikes = int(renting["num_bikes_available"].sum()), int(renting["num_ebikes_available"].sum())
kpis = [
    ("Bikes available", bikes, ""),
    ("E-bike share", 100 * ebikes / max(bikes, 1), "%"),
    (f"Empty stations<br>(capacity ≥ {MIN_CAPACITY})", int((large["state"] == "Empty").sum()), ""),
    (f"Full stations<br>(capacity ≥ {MIN_CAPACITY})", int((large["state"] == "Full").sum()), ""),
    ("Offline stations", int((stations["is_renting"] == 0).sum()), ""),
]
fig = make_subplots(rows=1, cols=len(kpis), specs=[[{"type": "indicator"}] * len(kpis)])
for i, (title, value, suffix) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(mode="number", value=value, title={"text": title, "font": {"size": 14}},
                               number={"suffix": suffix, "valueformat": ",.1f" if suffix else ",.0f"}), row=1, col=i)
fig.update_layout(template=TEMPLATE, height=190, margin=dict(t=50, b=10, l=10, r=10),
                  title=f"Network snapshot as of {as_of}")
fig.show()

## Where is availability low?

In [5]:
scatter = getattr(px, "scatter_map", None) or px.scatter_mapbox     # plotly >= 5.24 renamed this
fig = scatter(
    renting.assign(marker_size=renting["capacity"].clip(upper=40)),
    lat="lat", lon="lon", color="pct_full", size="marker_size", size_max=12, opacity=0.8,
    color_continuous_scale="RdYlBu", range_color=(0, 100), labels={"pct_full": "% full"},
    hover_name="name", hover_data={"num_bikes_available": True, "num_ebikes_available": True,
                                   "num_docks_available": True, "capacity": True,
                                   "marker_size": False, "lat": False, "lon": False},
    center={"lat": 41.88, "lon": -87.65}, zoom=10, height=620,
    title=f"Station availability: red = nearly empty, blue = nearly full ({len(renting):,} stations renting)",
)
fig.update_layout(**{("map_style" if hasattr(fig.layout, "map") else "mapbox_style"): "open-street-map"},
                  margin=dict(t=50, b=0, l=0, r=0))
fig.show()

## Rebalancing candidates
Large stations with no bikes to rent or no docks to return to.

In [6]:
problems = large[large["state"].isin(["Empty", "Full"])].nlargest(20, "capacity").sort_values("capacity")
fig = px.bar(problems, x="capacity", y="name", color="state", orientation="h",
             color_discrete_map={"Empty": "#d62728", "Full": "#1f77b4"}, template=TEMPLATE,
             height=max(300, 28 * len(problems) + 120), labels={"capacity": "Station capacity (docks)", "name": ""},
             title=f"Largest empty / full stations ({len(problems)} shown)")
fig.show()

## Availability over time

In [7]:
trend = q("""
SELECT snapshot_time,
       sum(num_bikes_available) - sum(num_ebikes_available)                      AS classic,
       sum(num_ebikes_available)                                                 AS ebike,
       count_if(num_bikes_available = 0 AND capacity >= {min_cap})               AS empty_stations,
       count_if(num_docks_available = 0 AND capacity >= {min_cap})               AS full_stations
FROM bike_db.v_station_health
WHERE dt = '{today}' AND is_renting = 1 AND {real}
GROUP BY snapshot_time ORDER BY snapshot_time
""", "trend")
trend["snapshot_time"] = pd.to_datetime(trend["snapshot_time"]).dt.tz_localize("UTC").dt.tz_convert(LOCAL_TZ)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Bikes available by type", f"Stations needing attention (capacity ≥ {MIN_CAPACITY})"))
for col, label, color in [("classic", "Classic bikes", "#1f77b4"), ("ebike", "E-bikes", "#2ca02c")]:
    fig.add_trace(go.Scatter(x=trend["snapshot_time"], y=trend[col], name=label, mode="lines+markers",
                             line_color=color), row=1, col=1)
for col, label, color in [("empty_stations", "Empty stations", "#d62728"), ("full_stations", "Full stations", "#ff7f0e")]:
    fig.add_trace(go.Scatter(x=trend["snapshot_time"], y=trend[col], name=label, mode="lines+markers",
                             line_color=color), row=2, col=1)
fig.update_layout(template=TEMPLATE, height=560, title=f"Today so far ({LOCAL_TZ} time)", legend_orientation="h")
fig.show()

## Which stations changed the most today?
First vs latest snapshot. Negative = draining (needs bikes), positive = filling (can donate).

In [8]:
delta = q("""
WITH ordered AS (
  SELECT station_id, name, capacity, num_bikes_available,
         row_number() OVER (PARTITION BY station_id ORDER BY ingest_ts ASC)  AS rn_first,
         row_number() OVER (PARTITION BY station_id ORDER BY ingest_ts DESC) AS rn_last
  FROM bike_db.v_station_health
  WHERE dt = '{today}' AND capacity >= {min_cap} AND {real}
)
SELECT f.name, l.num_bikes_available - f.num_bikes_available AS delta
FROM ordered f JOIN ordered l ON f.station_id = l.station_id
WHERE f.rn_first = 1 AND l.rn_last = 1
""", "drained")

window = f"{trend['snapshot_time'].min():%H:%M} to {trend['snapshot_time'].max():%H:%M}"
top = pd.concat([delta.nsmallest(10, "delta"), delta.nlargest(10, "delta")]).drop_duplicates("name")
top = top[top["delta"] != 0].sort_values("delta")
if top.empty:
    print("No change yet. Let the pipeline collect more snapshots.")
else:
    fig = px.bar(top, x="delta", y="name", orientation="h", color="delta", template=TEMPLATE,
                 color_continuous_scale="RdBu", color_continuous_midpoint=0, height=max(300, 28 * len(top) + 120),
                 labels={"delta": "Change in bikes", "name": ""}, title=f"Change in bikes available ({window})")
    fig.update_coloraxes(showscale=False)
    fig.show()

## Maintenance: disabled bikes and docks

In [9]:
maint = q("""
SELECT name, max(num_bikes_disabled) AS bikes_disabled, max(num_docks_disabled) AS docks_disabled
FROM bike_db.v_station_health
WHERE dt = '{today}' AND {real}
GROUP BY name
HAVING max(num_bikes_disabled) + max(num_docks_disabled) > 0
ORDER BY max(num_bikes_disabled) + max(num_docks_disabled) DESC
LIMIT 15
""", "maintenance")

long = maint.melt("name", var_name="type", value_name="count")
long["type"] = long["type"].map({"bikes_disabled": "Disabled bikes", "docks_disabled": "Disabled docks"})
fig = px.bar(long, x="count", y="name", color="type", orientation="h", template=TEMPLATE, height=520,
             category_orders={"name": maint["name"].tolist()[::-1]},
             color_discrete_map={"Disabled bikes": "#d62728", "Disabled docks": "#7f7f7f"},
             labels={"count": "Count (max seen today)", "name": ""}, title="Top 15 stations by out-of-service equipment")
fig.show()

## Stations that stopped reporting

In [10]:
stations = stations.assign(minutes_stale=(stations["ingest_ts"] - stations["last_reported"]) / 60)
stale = stations[stations["minutes_stale"].between(30, 7 * 24 * 60)]
dormant = int((stations["minutes_stale"] > 7 * 24 * 60).sum())

print(f"{len(stale)} stations silent for 30 min to 7 days; {dormant} silent for more than 7 days (likely decommissioned)")
stale.assign(hours_stale=(stale["minutes_stale"] / 60).round(1)).nlargest(15, "minutes_stale")[
    ["name", "capacity", "hours_stale"]].reset_index(drop=True)

21 stations silent for 30 min to 7 days; 4 silent for more than 7 days (likely decommissioned)


,name,capacity,hours_stale
0,Michigan Ave & 71st St,7,120.8
1,Damen Ave & 87th St,15,116.5
2,Stony Island Ave & 64th St,11,114.9
3,South Chicago Ave & 83rd St,11,105.8
4,Ashland Ave & 63rd St,11,96.5
5,Lamon Ave & Armitage Ave,15,72.8
6,Artesian Ave & 55th St,15,71.4
7,Cottage Grove Ave & 111th Pl,15,67.7
8,Dauphin Ave & 87th St,15,62.3
9,Kenosha Ave & Wellington Ave,15,45.9


## Cost: raw JSON vs curated Parquet
Same aggregation on both layers, restricted to the snapshots present in the Parquet table.

In [11]:
cur_max = q("SELECT max(ingest_ts) AS m FROM bike_db.station_status_curated WHERE dt = '{today}'", "cost_cursor").iloc[0]["m"]
if pd.isna(cur_max):
    raise RuntimeError("station_status_curated has no rows for today. Run sql/06_curated_parquet_ctas.sql, then re-run this cell.")

rows = []
for layer, tbl in [("Raw JSON", "bike_db.station_status"), ("Curated Parquet", "bike_db.station_status_curated")]:
    sql = (f"-- q:cost_{layer.split()[0].lower()}\nSELECT count(*) AS n, sum(num_bikes_available) AS bikes "
           f"FROM {tbl} WHERE dt = '{TODAY}' AND ingest_ts <= {int(cur_max)}")
    qid = wr.athena.start_query_execution(sql, database=DB, s3_output=OUTPUT, boto3_session=session)
    stats = wr.athena.wait_query(query_execution_id=qid, boto3_session=session)["Statistics"]
    n = int(wr.athena.get_query_results(query_execution_id=qid, boto3_session=session)["n"].iloc[0])
    rows.append({"layer": layer, "rows": n, "scanned_MB": round(stats["DataScannedInBytes"] / 1e6, 2),
                 "runtime_s": round(stats["EngineExecutionTimeInMillis"] / 1000, 2)})

cost = pd.DataFrame(rows)
if cost["rows"].nunique() > 1:
    print("WARNING: row counts differ, so the layers hold different data. Rebuild the Parquet table into a new folder.")
cost.assign(scan_vs_raw=(cost["scanned_MB"] / cost.loc[0, "scanned_MB"]).map("{:.1%}".format))

,layer,rows,scanned_MB,runtime_s,scan_vs_raw
0,Raw JSON,60291,29.07,0.54,100.0%
1,Curated Parquet,60291,0.03,0.48,0.1%
